In [2]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ==========================================
# 1. LOAD DATA & DEFINE FEATURES
# ==========================================
# Load the processed V3 features
df_cb = pd.read_csv("../data/processed/demand_features_v3.csv", parse_dates=["date"])

# Define all features
feature_cols_v3 = ['year', 'month', 'day_of_week', 'is_weekend', 'is_back_to_school', 'is_holiday_season', 
                   'lag_1', 'lag_7', 'rolling_mean_7', 'rolling_mean_30', 'is_promo', 'is_stockout']

# Explicitly tell CatBoost which features are categorical (not continuous numbers)
cat_features_indices = ['year', 'month', 'day_of_week', 'is_weekend', 'is_back_to_school', 'is_holiday_season', 'is_promo', 'is_stockout']

# Ensure categorical columns are cast to integers (CatBoost prefers int or string for categories)
for col in cat_features_indices:
    df_cb[col] = df_cb[col].astype(int)

all_results_cb = []
trained_models_cb = {}

# ==========================================
# 2. TRAIN & EVALUATE PER SKU
# ==========================================
for sku in df_cb["sku_id"].unique():
    sku_df = df_cb[df_cb["sku_id"] == sku].reset_index(drop=True)
    
    # 80/20 chronological split
    split_idx = int(len(sku_df) * 0.8)
    train_cb = sku_df.iloc[:split_idx]
    test_cb = sku_df.iloc[split_idx:]
    
    # Train on the differenced target (just like XGBoost)
    X_train_cb = train_cb[feature_cols_v3]
    y_train_diff_cb = train_cb["units_sold_diff"]
    
    X_test_cb = test_cb[feature_cols_v3]
    y_test_actual_cb = test_cb["units_sold"]
    test_lag_1_cb = test_cb["lag_1"] 
    
    # Initialize CatBoost Model
    # We use 200 iterations and a depth of 6 to match typical XGBoost default complexity
    model_cb = CatBoostRegressor(
        iterations=200, 
        learning_rate=0.05, 
        depth=6, 
        random_seed=42, 
        verbose=0 # Set to 0 to keep the notebook output clean
    )
    
    # Fit model, passing the categorical features explicitly
    model_cb.fit(X_train_cb, y_train_diff_cb, cat_features=cat_features_indices)
    
    # Predict the difference and reconstruct the actual predicted volume
    preds_diff_cb = model_cb.predict(X_test_cb)
    preds_actual_cb = test_lag_1_cb + preds_diff_cb
    
    # Prevent negative predictions
    preds_actual_cb = np.maximum(0, preds_actual_cb)
    
    # Calculate Metrics
    rmse = np.sqrt(mean_squared_error(y_test_actual_cb, preds_actual_cb))
    mae = mean_absolute_error(y_test_actual_cb, preds_actual_cb)
    mape = np.mean(np.abs((y_test_actual_cb - preds_actual_cb) / y_test_actual_cb)) * 100
    r2 = r2_score(y_test_actual_cb, preds_actual_cb)
    
    all_results_cb.append({
        "model": "CatBoost", 
        "sku_id": sku, 
        "RMSE": rmse, 
        "MAE": mae, 
        "MAPE": mape, 
        "R2": r2
    })
    
    trained_models_cb[sku] = model_cb

# ==========================================
# 3. SUMMARIZE RESULTS
# ==========================================
results_df_cb = pd.DataFrame(all_results_cb)
summary_df_cb = results_df_cb.groupby("model")[["RMSE", "MAE", "MAPE", "R2"]].mean().round(3)

print("\n--- CatBoost Performance Summary ---")
print(summary_df_cb)

# Save results
results_df_cb.to_csv("../data/processed/catboost_results_v3.csv", index=False)
summary_df_cb.to_csv("../data/processed/catboost_summary_v3.csv")


--- CatBoost Performance Summary ---
            RMSE     MAE   MAPE    R2
model                                
CatBoost  32.688  18.883  7.947  0.84


In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score

df_cb = pd.read_csv("../data/processed/demand_features_v4_1M.csv", parse_dates=["date"])

feature_cols = [
    'store_id', 'year', 'month', 'day_of_week', 'is_weekend', 
    'is_holiday_season', 'lag_1', 'lag_7', 'rolling_mean_7', 
    'rolling_mean_30', 'is_promo', 'is_stockout'
]

# Ensure categorical types for CatBoost
cat_features = ['store_id', 'year', 'month', 'day_of_week', 'is_weekend', 'is_holiday_season', 'is_promo', 'is_stockout']
for col in cat_features:
    df_cb[col] = df_cb[col].astype(str)

all_results_cb = []
for sku in df_cb["sku_id"].unique():
    sku_df = df_cb[df_cb["sku_id"] == sku].sort_values('date').reset_index(drop=True)
    split_idx = int(len(sku_df) * 0.8)
    
    train_cb, test_cb = sku_df.iloc[:split_idx], sku_df.iloc[split_idx:]
    X_train_cb, y_train_diff_cb = train_cb[feature_cols], train_cb["units_sold_diff"]
    X_test_cb, y_test_actual_cb, test_lag_1_cb = test_cb[feature_cols], test_cb["units_sold"], test_cb["lag_1"]
    
    model_cb = CatBoostRegressor(
        iterations=200, learning_rate=0.1, depth=6, random_seed=42, verbose=0, task_type="GPU"
    )
    
    model_cb.fit(X_train_cb, y_train_diff_cb, cat_features=cat_features)
    
    preds_diff_cb = model_cb.predict(X_test_cb)
    preds_actual_cb = np.maximum(0, test_lag_1_cb + preds_diff_cb)
    
    rmse = np.sqrt(mean_squared_error(y_test_actual_cb, preds_actual_cb))
    r2 = r2_score(y_test_actual_cb, preds_actual_cb)
    
    all_results_cb.append({"model": "CatBoost_1M", "sku_id": sku, "RMSE": rmse, "R2": r2})

print(pd.DataFrame(all_results_cb).groupby("model")[["RMSE", "R2"]].mean().round(3))

               RMSE     R2
model                     
CatBoost_1M  10.432  0.447
